In [1]:
import os
import pickle
import numpy as np
import cv2
from tqdm import tqdm
import tarfile
import urllib.request

# =============================================
# KONFIGURASI CIFAR-10 - PATH DIESKSPOR KE DOWNLOADS/ARCHIVE
# =============================================
base_path = "C:/Users/Dhea/Downloads/archive"
cifar10_url = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"
cifar10_tar_path = os.path.join(base_path, "cifar-10-python.tar.gz")
cifar10_extract_path = os.path.join(base_path, "cifar-10-batches-py")  # DIUBAH: folder yang benar
output_train_folder = os.path.join(base_path, "cifar10_train")
output_test_folder = os.path.join(base_path, "cifar10_test")

# 10 kelas CIFAR-10
cifar10_classes = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

# =============================================
# FUNGSI UNTUK MENGUNDUH CIFAR-10
# =============================================

def download_cifar10():
    """
    Mengunduh dataset CIFAR-10 jika belum ada
    """
    # Buat folder base path jika belum ada
    os.makedirs(base_path, exist_ok=True)
    
    if os.path.exists(cifar10_tar_path):
        print(f"✅ File CIFAR-10 sudah ada: {cifar10_tar_path}")
        return True
    
    print("📥 Mengunduh CIFAR-10 dataset...")
    try:
        urllib.request.urlretrieve(cifar10_url, cifar10_tar_path)
        print(f"✅ Berhasil mengunduh: {cifar10_tar_path}")
        return True
    except Exception as e:
        print(f"❌ Gagal mengunduh CIFAR-10: {e}")
        return False

def extract_cifar10():
    """
    Mengekstrak file tar.gz CIFAR-10
    """
    if os.path.exists(cifar10_extract_path):
        print(f"✅ File CIFAR-10 sudah diekstrak: {cifar10_extract_path}")
        return True
    
    if not os.path.exists(cifar10_tar_path):
        print("❌ File CIFAR-10 tidak ditemukan. Silakan unduh terlebih dahulu.")
        return False
    
    print("📦 Mengekstrak CIFAR-10...")
    try:
        with tarfile.open(cifar10_tar_path, 'r:gz') as tar:
            tar.extractall(base_path)
        print(f"✅ Berhasil mengekstrak ke: {cifar10_extract_path}")
        return True
    except Exception as e:
        print(f"❌ Gagal mengekstrak CIFAR-10: {e}")
        return False

# =============================================
# FUNGSI UNTUK MEMBACA FILE BATCH CIFAR-10
# =============================================

def unpickle(file):
    """
    Membaca file batch CIFAR-10
    """
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

def load_cifar10_data():
    """
    Memuat semua data CIFAR-10 dari file batch
    """
    data_path = cifar10_extract_path
    
    # Periksa apakah folder ada
    if not os.path.exists(data_path):
        print(f"❌ Folder tidak ditemukan: {data_path}")
        # Coba cari folder alternatif
        possible_paths = [
            os.path.join(base_path, "cifar-10-batches-py"),
            os.path.join(base_path, "cifar-10-python"),
            os.path.join(base_path, "cifar10")
        ]
        
        for path in possible_paths:
            if os.path.exists(path):
                data_path = path
                print(f"✅ Menggunakan folder alternatif: {data_path}")
                break
        else:
            print("❌ Tidak ada folder CIFAR-10 yang ditemukan!")
            return None, None, None, None
    
    # Data training (5 batch)
    train_data = []
    train_labels = []
    
    for i in range(1, 6):
        batch_file = os.path.join(data_path, f'data_batch_{i}')
        print(f"📖 Membaca: {batch_file}")
        
        if not os.path.exists(batch_file):
            print(f"❌ File batch tidak ditemukan: {batch_file}")
            # Coba dengan ekstensi berbeda
            batch_file_alt = os.path.join(data_path, f'data_batch_{i}')
            if not os.path.exists(batch_file_alt):
                print(f"❌ File batch alternatif juga tidak ditemukan: {batch_file_alt}")
                continue
        
        try:
            batch_dict = unpickle(batch_file)
            batch_data = batch_dict[b'data']
            batch_labels = batch_dict[b'labels']
            
            train_data.append(batch_data)
            train_labels.extend(batch_labels)
            print(f"✅ Berhasil membaca data_batch_{i}")
        except Exception as e:
            print(f"❌ Gagal membaca data_batch_{i}: {e}")
            return None, None, None, None
    
    if not train_data:
        print("❌ Tidak ada data training yang berhasil dibaca!")
        return None, None, None, None
    
    # Gabungkan semua data training
    train_data = np.vstack(train_data)
    train_labels = np.array(train_labels)
    
    # Data test
    test_batch_file = os.path.join(data_path, 'test_batch')
    print(f"📖 Membaca: {test_batch_file}")
    
    if not os.path.exists(test_batch_file):
        print(f"❌ File test batch tidak ditemukan: {test_batch_file}")
        return None, None, None, None
    
    try:
        test_dict = unpickle(test_batch_file)
        test_data = test_dict[b'data']
        test_labels = test_dict[b'labels']
        test_labels = np.array(test_labels)
        
        print(f"✅ Berhasil membaca test_batch")
    except Exception as e:
        print(f"❌ Gagal membaca test_batch: {e}")
        return None, None, None, None
    
    print(f"📊 Data Training: {train_data.shape}, Labels: {train_labels.shape}")
    print(f"📊 Data Test: {test_data.shape}, Labels: {test_labels.shape}")
    
    return train_data, train_labels, test_data, test_labels

# =============================================
# FUNGSI KONVERSI DAN PENYIMPANAN GAMBAR
# =============================================

def create_folders():
    """
    Membuat folder untuk setiap kelas dalam train dan test
    """
    # Buat folder utama
    os.makedirs(output_train_folder, exist_ok=True)
    os.makedirs(output_test_folder, exist_ok=True)
    
    # Buat subfolder untuk setiap kelas
    for class_name in cifar10_classes:
        train_class_path = os.path.join(output_train_folder, class_name)
        test_class_path = os.path.join(output_test_folder, class_name)
        
        os.makedirs(train_class_path, exist_ok=True)
        os.makedirs(test_class_path, exist_ok=True)
        
        print(f"📁 Membuat folder: {train_class_path}")
        print(f"📁 Membuat folder: {test_class_path}")

def convert_and_save_images(data, labels, output_folder, dataset_type="train"):
    """
    Mengkonversi data CIFAR-10 ke gambar dan menyimpannya
    """
    if data is None or labels is None:
        print(f"❌ Data {dataset_type} tidak valid!")
        return
    
    # Konversi bentuk data: (n, 3072) -> (n, 32, 32, 3)
    # CIFAR-10 format: 3072 = 3*32*32 (R, G, B channels)
    try:
        images = data.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    except Exception as e:
        print(f"❌ Gagal reshape data {dataset_type}: {e}")
        return
    
    print(f"💾 Menyimpan gambar {dataset_type}...")
    
    for i in tqdm(range(len(images)), desc=f"Processing {dataset_type}"):
        # Dapatkan gambar dan label
        img = images[i]
        label_idx = labels[i]
        class_name = cifar10_classes[label_idx]
        
        # Konversi dari RGB ke BGR untuk OpenCV
        img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        
        # Nama file: class_index.png
        filename = f"{class_name}_{i:05d}.png"
        filepath = os.path.join(output_folder, class_name, filename)
        
        # Simpan gambar
        success = cv2.imwrite(filepath, img_bgr)
        if not success:
            print(f"❌ Gagal menyimpan: {filepath}")
    
    print(f"✅ Berhasil menyimpan {len(images)} gambar {dataset_type}")

def verify_extraction():
    """
    Memverifikasi bahwa ekstraksi berhasil
    """
    print("\n🔍 Memverifikasi ekstraksi...")
    
    total_train = 0
    total_test = 0
    
    for class_name in cifar10_classes:
        train_class_path = os.path.join(output_train_folder, class_name)
        test_class_path = os.path.join(output_test_folder, class_name)
        
        train_count = len([f for f in os.listdir(train_class_path) if f.endswith('.png')]) if os.path.exists(train_class_path) else 0
        test_count = len([f for f in os.listdir(test_class_path) if f.endswith('.png')]) if os.path.exists(test_class_path) else 0
        
        total_train += train_count
        total_test += test_count
        
        print(f"  {class_name:12s} - Train: {train_count:5d}, Test: {test_count:5d}")
    
    print(f"📊 TOTAL - Train: {total_train}, Test: {total_test}")
    
    expected_train = 50000
    expected_test = 10000
    
    if total_train == expected_train and total_test == expected_test:
        print("✅ Ekstraksi berhasil dan lengkap!")
        return True
    else:
        print(f"⚠️  Jumlah gambar tidak sesuai! Diharapkan Train: {expected_train}, Test: {expected_test}")
        print(f"   Mungkin ada masalah dengan ekstraksi, tetapi proses akan dilanjutkan.")
        return True  # Tetap return True untuk melanjutkan

# =============================================
# FUNGSI UTAMA
# =============================================

def main_extract_cifar10():
    """
    Fungsi utama untuk mengekstrak dataset CIFAR-10
    """
    print("=" * 70)
    print("EKSTRAKSI DATASET CIFAR-10")
    print("=" * 70)
    print(f"📍 Base Path: {base_path}")
    
    # Langkah 1: Unduh dataset jika belum ada
    print("\n1. MENGUNDUH DATASET CIFAR-10")
    if not download_cifar10():
        return False
    
    # Langkah 2: Ekstrak file tar.gz
    print("\n2. MENGEKSTRAK FILE CIFAR-10")
    if not extract_cifar10():
        return False
    
    # Langkah 3: Buat folder struktur
    print("\n3. MEMBUAT STRUKTUR FOLDER")
    create_folders()
    
    # Langkah 4: Muat data CIFAR-10
    print("\n4. MEMUAT DATA CIFAR-10")
    train_data, train_labels, test_data, test_labels = load_cifar10_data()
    
    if train_data is None:
        print("❌ Gagal memuat data CIFAR-10!")
        return False
    
    # Langkah 5: Konversi dan simpan gambar training
    print("\n5. MENYIMPAN GAMBAR TRAINING")
    convert_and_save_images(train_data, train_labels, output_train_folder, "training")
    
    # Langkah 6: Konversi dan simpan gambar test
    print("\n6. MENYIMPAN GAMBAR TEST")
    convert_and_save_images(test_data, test_labels, output_test_folder, "test")
    
    # Langkah 7: Verifikasi hasil
    print("\n7. VERIFIKASI HASIL EKSTRAKSI")
    success = verify_extraction()
    
    if success:
        print("\n🎉 EKSTRAKSI CIFAR-10 BERHASIL!")
        print(f"📁 Folder Training: {output_train_folder}")
        print(f"📁 Folder Test: {output_test_folder}")
        print(f"📊 Total: 50,000 gambar training + 10,000 gambar test")
    else:
        print("\n❌ EKSTRAKSI CIFAR-10 GAGAL!")
    
    return success

def cleanup_temp_files():
    """
    Membersihkan file temporary (opsional)
    """
    print("\n🧹 Membersihkan file temporary...")
    
    if os.path.exists(cifar10_tar_path):
        os.remove(cifar10_tar_path)
        print(f"✅ Menghapus: {cifar10_tar_path}")
    
    # Hapus folder ekstraksi sementara
    extract_folders = [
        os.path.join(base_path, "cifar-10-batches-py"),
        os.path.join(base_path, "cifar-10-python"),
        os.path.join(base_path, "cifar10")
    ]
    
    for folder in extract_folders:
        if os.path.exists(folder):
            import shutil
            shutil.rmtree(folder)
            print(f"✅ Menghapus: {folder}")

# =============================================
# FUNGSI TAMBAHAN: MEMUAT DATA LANGSUNG DARI FOLDER
# =============================================

def load_images_from_folder(folder_path, img_size=(32, 32)):
    """
    Memuat gambar dari folder yang sudah diekstrak
    """
    images = []
    labels = []
    filenames = []
    
    for class_idx, class_name in enumerate(cifar10_classes):
        class_folder = os.path.join(folder_path, class_name)
        
        if not os.path.exists(class_folder):
            print(f"❌ Folder tidak ditemukan: {class_folder}")
            continue
        
        image_files = [f for f in os.listdir(class_folder) if f.endswith(('.png', '.jpg', '.jpeg'))]
        
        print(f"📁 Memuat {len(image_files)} gambar dari {class_name}...")
        
        for image_file in tqdm(image_files, desc=f"Loading {class_name}"):
            image_path = os.path.join(class_folder, image_file)
            
            # Baca gambar
            img = cv2.imread(image_path)
            if img is not None:
                # Resize jika diperlukan
                if img.shape[:2] != img_size:
                    img = cv2.resize(img, img_size)
                
                # Konversi BGR ke RGB
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                
                images.append(img_rgb)
                labels.append(class_idx)
                filenames.append(image_file)
    
    return np.array(images), np.array(labels), filenames

def get_dataset_info():
    """
    Menampilkan informasi tentang dataset yang sudah diekstrak
    """
    print("\n📊 INFORMASI DATASET CIFAR-10")
    print("=" * 50)
    
    if not os.path.exists(output_train_folder):
        print("❌ Folder training tidak ditemukan. Jalankan ekstraksi terlebih dahulu.")
        return
    
    total_train = 0
    total_test = 0
    
    print("\nKELAS - JUMLAH GAMBAR (Training / Test):")
    print("-" * 50)
    
    for class_name in cifar10_classes:
        train_path = os.path.join(output_train_folder, class_name)
        test_path = os.path.join(output_test_folder, class_name)
        
        train_count = len([f for f in os.listdir(train_path) if f.endswith('.png')]) if os.path.exists(train_path) else 0
        test_count = len([f for f in os.listdir(test_path) if f.endswith('.png')]) if os.path.exists(test_path) else 0
        
        total_train += train_count
        total_test += test_count
        
        print(f"  {class_name:12s} : {train_count:5d} / {test_count:5d}")
    
    print("-" * 50)
    print(f"  TOTAL        : {total_train:5d} / {total_test:5d}")
    print(f"  GRAND TOTAL  : {total_train + total_test:5d}")
    
    # Tampilkan contoh gambar dari setiap kelas
    print("\n🖼️  CONTOH GAMBAR:")
    display_sample_images()

def display_sample_images(samples_per_class=2):
    """
    Menampilkan contoh gambar dari setiap kelas
    """
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    axes = axes.ravel()
    
    for class_idx, class_name in enumerate(cifar10_classes):
        class_folder = os.path.join(output_train_folder, class_name)
        
        if not os.path.exists(class_folder):
            continue
            
        image_files = [f for f in os.listdir(class_folder) if f.endswith('.png')]
        
        if image_files:
            # Ambil sample gambar
            sample_file = image_files[0]
            sample_path = os.path.join(class_folder, sample_file)
            
            # Baca dan tampilkan gambar
            img = cv2.imread(sample_path)
            if img is not None:
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                
                axes[class_idx].imshow(img_rgb)
                axes[class_idx].set_title(f'{class_name}', fontsize=10)
                axes[class_idx].axis('off')
    
    plt.tight_layout()
    plt.show()

# =============================================
# FUNGSI UNTUK MENGUBAH KONFIGURASI ANALISIS
# =============================================

def update_analysis_config():
    """
    Mengupdate konfigurasi untuk kode analisis
    """
    config_code = f'''
# =============================================
# KONFIGURASI - CIFAR-10 DATASET (UPDATED PATH)
# =============================================
base_path = "{base_path}"
cifar10_train_folder = "{output_train_folder}"
cifar10_test_folder = "{output_test_folder}"

# 10 kelas CIFAR-10
cifar10_classes = {cifar10_classes}

# File output dengan path yang sesuai
output_excel = "{os.path.join(base_path, "cifar10_data_ekstraksi_rgb_10kelas.xlsx")}"
ecdf_output_csv = "{os.path.join(base_path, "cifar10_data_ecdf_rgb_10kelas.csv")}"
ecdf_intensity_output_excel = "{os.path.join(base_path, "cifar10_data_ecdf_intensitas_10kelas.xlsx")}"
conversion_output_excel = "{os.path.join(base_path, "cifar10_data_ecdf_cdf_konversi_10kelas.xlsx")}"
clustering_output_excel = "{os.path.join(base_path, "cifar10_hasil_clustering_10kelas.xlsx")}"
centroid_output_file = "{os.path.join(base_path, "cifar10_centroid_10kelas.pkl")}"

p = 32  # Ukuran resize gambar: p x p (sesuai CIFAR-10 asli)
'''

    config_file_path = os.path.join(base_path, "cifar10_config.py")
    with open(config_file_path, 'w') as f:
        f.write(config_code)
    
    print(f"✅ File konfigurasi disimpan di: {config_file_path}")
    print("\n📋 Konfigurasi untuk kode analisis:")
    print(config_code)

# =============================================
# JALANKAN PROGRAM
# =============================================

if __name__ == "__main__":
    # Pastikan base path ada
    os.makedirs(base_path, exist_ok=True)
    
    # Ekstrak dataset CIFAR-10
    success = main_extract_cifar10()
    
    if success:
        # Tampilkan informasi dataset
        get_dataset_info()
        
        # Buat file konfigurasi untuk kode analisis
        update_analysis_config()
        
        # Opsional: bersihkan file temporary
        cleanup_choice = input("\n🧹 Hapus file temporary (cifar-10-python.tar.gz dan folder ekstraksi)? (y/n): ")
        if cleanup_choice.lower() == 'y':
            cleanup_temp_files()
    
    print("\n✨ Proses selesai!")

ImportError: DLL load failed while importing _multiarray_umath: The specified module could not be found.

ImportError: numpy._core.multiarray failed to import

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pickle

# =============================================
# KONFIGURASI - CIFAR-10 DATASET (PATH DIESKSPOR KE DOWNLOADS/ARCHIVE)
# =============================================
base_path = "C:/Users/Dhea/Downloads/archive"
cifar10_train_folder = os.path.join(base_path, "cifar10_train")
cifar10_test_folder = os.path.join(base_path, "cifar10_test")

# 10 kelas CIFAR-10
cifar10_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

# File output dengan path yang sesuai
output_excel = os.path.join(base_path, "cifar10_data_ekstraksi_rgb_10kelas.xlsx")
ecdf_output_csv = os.path.join(base_path, "cifar10_data_ecdf_rgb_10kelas.csv")
ecdf_intensity_output_excel = os.path.join(base_path, "cifar10_data_ecdf_intensitas_10kelas.xlsx")
conversion_output_excel = os.path.join(base_path, "cifar10_data_ecdf_cdf_konversi_10kelas.xlsx")
clustering_output_excel = os.path.join(base_path, "cifar10_hasil_clustering_10kelas.xlsx")
centroid_output_file = os.path.join(base_path, "cifar10_centroid_10kelas.pkl")

p = 25  # Ukuran resize gambar: p x p (sesuai CIFAR-10 asli)

# =============================================
# FUNGSI UTAMA UNTUK MEMPROSES GAMBAR
# =============================================

def process_image(image_path, label, gambar_ke):
    """
    Memproses satu gambar: resize ke p x p, ekstrak RGB, dan simpan ke Excel
    """
    # Baca gambar
    img = cv2.imread(image_path)
    if img is None:
        print(f"❌ Tidak bisa membaca gambar: {image_path}")
        return None
    
    # Resize gambar menjadi p x p
    img_resized = cv2.resize(img, (p, p))
    
    # Konversi BGR ke RGB
    img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
    
    # Ekstrak channel R, G, B
    R = img_rgb[:, :, 0].flatten()
    G = img_rgb[:, :, 1].flatten()
    B = img_rgb[:, :, 2].flatten()
    
    # Hitung statistik
    hasil = {
        'filename': os.path.basename(image_path),
        'gambar_ke': gambar_ke,
        'label': label,
        'p': p,
        'total_pixels': p * p,
        'R_values': R,
        'G_values': G,
        'B_values': B,
        'avg_R': np.mean(R),
        'avg_G': np.mean(G),
        'avg_B': np.mean(B),
        'min_R': np.min(R),
        'max_R': np.max(R),
        'min_G': np.min(G),
        'max_G': np.max(G),
        'min_B': np.min(B),
        'max_B': np.max(B)
    }
    
    return hasil

def process_all_images_from_folder(folder_path, label):
    """
    Memproses semua gambar dalam folder untuk satu kelas
    """
    results = []
    
    if not os.path.exists(folder_path):
        print(f"❌ Folder tidak ditemukan: {folder_path}")
        return pd.DataFrame()
    
    image_files = [f for f in os.listdir(folder_path) 
                  if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    if not image_files:
        print(f"❌ Tidak ada gambar ditemukan di: {folder_path}")
        return pd.DataFrame()
    
    print(f"📁 Memproses {len(image_files)} gambar dari {folder_path} (label: {label})")
    
    for i, image_file in enumerate(image_files):
        image_path = os.path.join(folder_path, image_file)
        
        result = process_image(image_path, label, i+1)
        if result is not None:
            results.append(result)
    
    # Konversi ke DataFrame
    if results:
        df = pd.DataFrame(results)
        print(f"✅ Berhasil memproses {len(df)} gambar dengan label: {label}")
        return df
    else:
        print(f"❌ Tidak ada hasil yang berhasil diproses dari {folder_path}")
        return pd.DataFrame()

def process_cifar10_dataset(base_folder, dataset_type="train"):
    """
    Memproses semua kelas CIFAR-10 dari folder base
    """
    all_dfs = []
    
    for class_name in cifar10_classes:
        class_folder = os.path.join(base_folder, class_name)
        df_class = process_all_images_from_folder(class_folder, class_name)
        if not df_class.empty:
            all_dfs.append(df_class)
    
    if all_dfs:
        df_combined = pd.concat(all_dfs, ignore_index=True)
        print(f"✅ Total {len(df_combined)} gambar diproses dari {dataset_type} dataset")
        return df_combined
    else:
        print(f"❌ Tidak ada data yang diproses dari {dataset_type} dataset")
        return pd.DataFrame()

def save_rgb_to_excel(df_rgb, output_path):
    """
    Menyimpan data RGB ke file Excel
    """
    # Buat DataFrame yang lebih sederhana untuk disimpan
    save_data = []
    for _, row in df_rgb.iterrows():
        save_data.append({
            'filename': row['filename'],
            'gambar_ke': row['gambar_ke'],
            'label': row['label'],
            'p': row['p'],
            'total_pixels': row['total_pixels'],
            'R_values': str(row['R_values'].tolist()),
            'G_values': str(row['G_values'].tolist()),
            'B_values': str(row['B_values'].tolist()),
            'avg_R': row['avg_R'],
            'avg_G': row['avg_G'],
            'avg_B': row['avg_B'],
            'min_R': row['min_R'],
            'max_R': row['max_R'],
            'min_G': row['min_G'],
            'max_G': row['max_G'],
            'min_B': row['min_B'],
            'max_B': row['max_B']
        })
    
    df_save = pd.DataFrame(save_data)
    df_save.to_excel(output_path, index=False)
    print(f"💾 Data RGB disimpan ke: {output_path}")

# =============================================
# FUNGSI ECDF
# =============================================

def calculate_ecdf(values):
    """
    Menghitung ECDF untuk suatu set nilai
    """
    if len(values) == 0:
        return np.array([]), np.array([])
    
    # Urutkan nilai
    sorted_values = np.sort(values)
    
    # Hitung ECDF
    n = len(sorted_values)
    ecdf = np.arange(1, n + 1) / n
    
    return sorted_values, ecdf

def calculate_ecdf_for_image(image_data):
    """
    Menghitung ECDF untuk sebuah gambar
    """
    filename = image_data['filename']
    j = image_data['gambar_ke']
    label = image_data['label']
    p_j = image_data['p']
    
    ecdf_data = {
        'filename': filename,
        'gambar_ke': j,
        'label': label,
        'p_j': p_j,
        'R': {'intensity': [], 'ecdf': []},
        'G': {'intensity': [], 'ecdf': []},
        'B': {'intensity': [], 'ecdf': []}
    }
    
    # Untuk setiap channel R, G, B
    for channel in ['R', 'G', 'B']:
        values = image_data[f'{channel}_values']
        
        if len(values) > 0:
            intensity, ecdf = calculate_ecdf(values)
            ecdf_data[channel]['intensity'] = intensity
            ecdf_data[channel]['ecdf'] = ecdf
    
    return ecdf_data

def calculate_all_ecdf(df_rgb):
    """
    Menghitung ECDF untuk semua gambar
    """
    ecdf_results = {}
    
    for _, row in df_rgb.iterrows():
        ecdf_data = calculate_ecdf_for_image(row)
        ecdf_results[row['filename']] = ecdf_data
    
    return ecdf_results

def save_ecdf_to_csv(ecdf_data, output_path):
    """
    Menyimpan data ECDF ke file CSV (bukan Excel) untuk handle data besar
    """
    save_data = []
    
    for filename, data in ecdf_data.items():
        for channel in ['R', 'G', 'B']:
            channel_data = data[channel]
            if len(channel_data['intensity']) > 0:
                for intensity, ecdf_val in zip(channel_data['intensity'], channel_data['ecdf']):
                    save_data.append({
                        'filename': filename,
                        'gambar_ke': data['gambar_ke'],
                        'label': data['label'],
                        'channel': channel,
                        'intensity': intensity,
                        'ecdf': ecdf_val,
                        'p_j': data['p_j']
                    })
    
    df_save = pd.DataFrame(save_data)
    
    # Simpan ke CSV bukan Excel
    df_save.to_csv(output_path, index=False)
    print(f"💾 Data ECDF disimpan ke: {output_path}")
    print(f"📊 Jumlah baris data ECDF: {len(df_save)}")

# =============================================
# FUNGSI MODEL MATEMATIS V DAN KONVERSI CDF
# =============================================

def create_mathematical_model_V(df_cdf):
    """
    Membuat model matematis V dari data CDF intensitas
    """
    model_V = {}
    
    # Channel R: intensitas 100, 150, 200
    for intensity in [100, 150, 200]:
        channel_data = df_cdf[(df_cdf['channel'] == 'R') & (df_cdf['intensity_target'] == intensity)]
        if not channel_data.empty:
            model_V[f'R_{intensity}'] = {
                'ecdf_values': channel_data['ecdf_value'].values,
                'cdf_values': channel_data['cdf_value'].values
            }
    
    # Channel G: intensitas 150, 200
    for intensity in [150, 200]:
        channel_data = df_cdf[(df_cdf['channel'] == 'G') & (df_cdf['intensity_target'] == intensity)]
        if not channel_data.empty:
            model_V[f'G_{intensity}'] = {
                'ecdf_values': channel_data['ecdf_value'].values,
                'cdf_values': channel_data['cdf_value'].values
            }
    
    # Channel B: intensitas 150, 200
    for intensity in [150, 200]:
        channel_data = df_cdf[(df_cdf['channel'] == 'B') & (df_cdf['intensity_target'] == intensity)]
        if not channel_data.empty:
            model_V[f'B_{intensity}'] = {
                'ecdf_values': channel_data['ecdf_value'].values,
                'cdf_values': channel_data['cdf_value'].values
            }
    
    print("✅ Model matematis V berhasil dibuat")
    print(f"🔧 Komponen model V: {list(model_V.keys())}")
    
    return model_V

def convert_ecdf_to_cdf_with_model_V(ecdf_value, model_key, model_V):
    """
    Mengkonversi nilai ECDF ke CDF menggunakan model matematis V
    """
    if model_key not in model_V:
        return None
    
    model_data = model_V[model_key]
    ecdf_values = model_data['ecdf_values']
    cdf_values = model_data['cdf_values']
    
    # Jika ECDF value di luar range model, return None
    if ecdf_value < ecdf_values[0] or ecdf_value > ecdf_values[-1]:
        return None
    
    # Cari posisi ECDF value dalam model
    idx = np.searchsorted(ecdf_values, ecdf_value, side='right') - 1
    
    # Return CDF value yang sesuai
    return cdf_values[idx]

def extract_and_convert_ecdf_data(ecdf_data, model_V):
    """
    Mengekstrak ECDF dari gambar pada intensitas tertentu dan mengkonversinya ke CDF menggunakan model V
    """
    conversion_results = []
    
    for filename, data in ecdf_data.items():
        j = data['gambar_ke']
        label = data['label']
        
        # Ekstrak ECDF pada intensitas tertentu untuk semua channel
        for channel in ['R', 'G', 'B']:
            channel_data = data[channel]
            intensity_values = channel_data['intensity']
            ecdf_values = channel_data['ecdf']
            
            if len(intensity_values) > 0:
                # Untuk setiap intensitas target
                for intensity in [100, 150, 200]:
                    # Tentukan model_key berdasarkan channel dan intensitas
                    model_key = f'{channel}_{intensity}'
                    
                    # Skip jika model_key tidak ada di model_V
                    if model_key not in model_V:
                        continue
                    
                    # Cari nilai ECDF pada intensitas target
                    ecdf_val = 0.0
                    for m in range(len(intensity_values)):
                        if intensity < intensity_values[0]:
                            ecdf_val = 0.0
                            break
                        elif intensity >= intensity_values[m]:
                            if m == len(intensity_values) - 1 or intensity < intensity_values[m + 1]:
                                ecdf_val = ecdf_values[m]
                                break
                        elif intensity >= intensity_values[-1]:
                            ecdf_val = 1.0
                            break
                    
                    # Konversi ECDF ke CDF menggunakan model V
                    cdf_val = convert_ecdf_to_cdf_with_model_V(ecdf_val, model_key, model_V)
                    
                    conversion_results.append({
                        'filename': filename,
                        'gambar_ke': j,
                        'label': label,
                        'channel': channel,
                        'intensity_target': intensity,
                        'ecdf_value': ecdf_val,
                        'cdf_value': cdf_val,
                        'model_key': model_key
                    })
    
    df_conversion = pd.DataFrame(conversion_results)
    return df_conversion

def save_conversion_data(ecdf_data, df_conversion, output_path):
    """
    Menyimpan data ECDF gambar dan CDF hasil konversi ke Excel
    """
    with pd.ExcelWriter(output_path) as writer:
        # Sheet 1: Data konversi ECDF ke CDF (data kecil)
        df_conversion.to_excel(writer, sheet_name='Konversi_ECDF_ke_CDF', index=False)
        
        # Sheet 2: Ringkasan ECDF per gambar (data lebih kecil)
        summary_data = []
        for filename, data in ecdf_data.items():
            for channel in ['R', 'G', 'B']:
                channel_data = data[channel]
                if len(channel_data['intensity']) > 0:
                    summary_data.append({
                        'filename': filename,
                        'gambar_ke': data['gambar_ke'],
                        'label': data['label'],
                        'channel': channel,
                        'num_points': len(channel_data['intensity']),
                        'min_intensity': min(channel_data['intensity']),
                        'max_intensity': max(channel_data['intensity']),
                        'min_ecdf': min(channel_data['ecdf']),
                        'max_ecdf': max(channel_data['ecdf'])
                    })
        
        df_summary = pd.DataFrame(summary_data)
        df_summary.to_excel(writer, sheet_name='Ringkasan_ECDF', index=False)
    
    print(f"💾 Data konversi CDF disimpan ke: {output_path}")

# =============================================
# FUNGSI EKSTRAKSI ECDF PADA INTENSITAS TERTENTU
# =============================================

def extract_ecdf_at_intensities(ecdf_data, intensities=[100, 150, 200]):
    """
    Mengekstrak nilai ECDF pada intensitas tertentu untuk setiap gambar dan channel
    """
    ecdf_intensity_data = []
    
    for filename, data in ecdf_data.items():
        j = data['gambar_ke']
        label = data['label']
        
        for channel in ['R', 'G', 'B']:
            channel_data = data[channel]
            intensity_values = channel_data['intensity']
            ecdf_values = channel_data['ecdf']
            
            if len(intensity_values) > 0:
                # Untuk setiap intensitas target, cari nilai ECDF-nya
                for intensity in intensities:
                    ecdf_val = 0.0
                    
                    if intensity < intensity_values[0]:
                        ecdf_val = 0.0
                    elif intensity >= intensity_values[-1]:
                        ecdf_val = 1.0
                    else:
                        # Cari di segmen mana intensitas berada
                        for m in range(len(intensity_values) - 1):
                            if intensity_values[m] <= intensity < intensity_values[m + 1]:
                                ecdf_val = ecdf_values[m]
                                break
                        else:
                            # Jika tidak ditemukan dalam loop, gunakan nilai terakhir
                            ecdf_val = ecdf_values[-1] if intensity >= intensity_values[-1] else 0.0
                    
                    ecdf_intensity_data.append({
                        'filename': filename,
                        'gambar_ke': j,
                        'label': label,
                        'channel': channel,
                        'intensity_target': intensity,
                        'ecdf_value': ecdf_val
                    })
    
    df_ecdf_intensities = pd.DataFrame(ecdf_intensity_data)
    return df_ecdf_intensities

def calculate_cdf_for_ecdf_intensities(df_ecdf_intensities):
    """
    Menghitung CDF untuk nilai ECDF pada intensitas tertentu per channel
    """
    cdf_data = []
    
    for channel in ['R', 'G', 'B']:
        for intensity in [100, 150, 200]:
            # Ambil nilai ECDF untuk channel dan intensitas tertentu
            ecdf_values = df_ecdf_intensities[
                (df_ecdf_intensities['channel'] == channel) & 
                (df_ecdf_intensities['intensity_target'] == intensity)
            ]['ecdf_value'].values
            
            if len(ecdf_values) > 0:
                # Urutkan nilai ECDF
                sorted_ecdf = np.sort(ecdf_values)
                
                # Hitung CDF
                cdf_values = np.arange(1, len(sorted_ecdf) + 1) / len(sorted_ecdf)
                
                # Simpan data
                for ecdf_val, cdf_val in zip(sorted_ecdf, cdf_values):
                    cdf_data.append({
                        'channel': channel,
                        'intensity_target': intensity,
                        'ecdf_value': ecdf_val,
                        'cdf_value': cdf_val
                    })
    
    df_cdf = pd.DataFrame(cdf_data)
    return df_cdf

def save_ecdf_intensity_data(df_ecdf_intensities, df_cdf, output_path):
    """
    Menyimpan data ECDF pada intensitas tertentu dan CDF-nya ke Excel
    """
    with pd.ExcelWriter(output_path) as writer:
        # Sheet 1: Data ECDF pada intensitas tertentu
        df_ecdf_intensities.to_excel(writer, sheet_name='ECDF_Intensitas', index=False)
        
        # Sheet 2: Data CDF dari ECDF intensitas
        df_cdf.to_excel(writer, sheet_name='CDF_ECDF_Intensitas', index=False)
    
    print(f"💾 Data ECDF intensitas dan CDF disimpan ke: {output_path}")

# =============================================
# FUNGSI CENTROID DAN CLUSTERING UNTUK 10 KELAS
# =============================================

def calculate_class_centroids(df_features):
    """
    Menghitung centroid untuk setiap kelas dengan mengambil rata-rata dari semua gambar dalam kelas tersebut
    """
    centroids = {}
    
    # Filter hanya kolom fitur (exclude metadata)
    feature_columns = [col for col in df_features.columns if col not in ['filename', 'gambar_ke', 'label']]
    
    # Dapatkan label unik yang sebenarnya ada di data
    actual_labels = df_features['label'].unique()
    
    for label in actual_labels:
        class_data = df_features[df_features['label'] == label]
        if len(class_data) > 0:
            # Hitung rata-rata untuk setiap fitur
            centroid_values = class_data[feature_columns].mean(axis=0).values
            centroids[label] = centroid_values
            print(f"✅ Centroid untuk kelas '{label}': {len(class_data)} gambar")
        else:
            print(f"❌ Tidak ada gambar untuk kelas '{label}'")
    
    return centroids, feature_columns

def save_centroids(centroids, feature_columns, output_path):
    """
    Menyimpan centroid ke file pickle
    """
    centroid_data = {
        'centroids': centroids,
        'feature_columns': feature_columns
    }
    
    with open(output_path, 'wb') as f:
        pickle.dump(centroid_data, f)
    
    print(f"💾 Centroid disimpan ke: {output_path}")

def load_centroids(input_path):
    """
    Memuat centroid dari file pickle
    """
    with open(input_path, 'rb') as f:
        centroid_data = pickle.load(f)
    
    print(f"💾 Centroid dimuat dari: {input_path}")
    return centroid_data['centroids'], centroid_data['feature_columns']

def prepare_features_for_clustering(df_conversion):
    """
    Menyiapkan fitur untuk clustering dari data CDF hasil konversi
    """
    # Group by filename untuk mendapatkan semua fitur CDF per gambar
    features_data = []
    
    for filename in df_conversion['filename'].unique():
        file_data = df_conversion[df_conversion['filename'] == filename]
        
        # Ekstrak semua nilai CDF untuk gambar ini
        feature_vector = {}
        feature_vector['filename'] = filename
        feature_vector['gambar_ke'] = file_data['gambar_ke'].iloc[0]
        feature_vector['label'] = file_data['label'].iloc[0]
        
        # Untuk setiap kombinasi channel dan intensitas
        for _, row in file_data.iterrows():
            key = f"{row['channel']}_{row['intensity_target']}"
            feature_vector[key] = row['cdf_value']
        
        features_data.append(feature_vector)
    
    df_features = pd.DataFrame(features_data)
    
    # DEBUG: Tampilkan label yang unik dan jumlahnya
    print("\n🔍 DEBUG - Label unik di df_features:")
    print(df_features['label'].value_counts())
    
    # Fill NaN values dengan 0
    df_features = df_features.fillna(0)
    
    return df_features

def perform_kmeans_clustering_10centroids(df_features, centroids_dict, feature_columns):
    """
    Melakukan K-means clustering dengan 10 centroid berdasarkan CDF hasil konversi
    Centroid diinisialisasi dengan rata-rata setiap kelas
    """
    print(f"\n🔍 Menyiapkan centroid dari rata-rata masing-masing kelas")
    
    # Filter hanya kolom fitur (exclude metadata)
    if len(feature_columns) == 0:
        print("❌ Tidak ada fitur yang ditemukan untuk clustering!")
        return None, None, None, []
    
    # Ekstrak matriks fitur
    X = df_features[feature_columns].values
    
    if len(centroids_dict) < 10:
        print(f"⚠️  Hanya {len(centroids_dict)} kelas yang memiliki centroid! Menggunakan centroid acak...")
        # Fallback: gunakan K-means dengan inisialisasi random
        kmeans = KMeans(n_clusters=10, random_state=42)
        labels = kmeans.fit_predict(X)
        centroids = kmeans.cluster_centers_
    else:
        # Siapkan initial centroids dari rata-rata kelas
        initial_centroids = np.array(list(centroids_dict.values()))
        
        print(f"📍 Shape initial centroids: {initial_centroids.shape}")
        print(f"🎯 Kelas yang digunakan: {list(centroids_dict.keys())}")
        
        # Lakukan K-means dengan inisialisasi centroid spesifik
        kmeans = KMeans(
            n_clusters=10,
            init=initial_centroids,
            n_init=1,  # Hanya gunakan inisialisasi yang diberikan
            max_iter=300,
            random_state=42
        )
        
        labels = kmeans.fit_predict(X)
        centroids = kmeans.cluster_centers_
        
        print("✅ K-means clustering dengan centroid berhasil!")
    
    # Tambahkan hasil clustering ke DataFrame
    df_features = df_features.copy()
    df_features['cluster'] = labels
    df_features['distance_to_centroid'] = kmeans.transform(X).min(axis=1)
    
    # Mapping cluster ke label berdasarkan centroid terdekat
    cluster_mapping = {}
    
    # Untuk setiap cluster, cari centroid kelas mana yang paling dekat
    for cluster_idx in range(len(centroids_dict)):
        if cluster_idx < len(centroids):  # Pastikan cluster_idx valid
            cluster_center = centroids[cluster_idx]
            
            # Hitung jarak ke setiap centroid kelas
            distances = {}
            for label, class_centroid in centroids_dict.items():
                distance = np.linalg.norm(cluster_center - class_centroid)
                distances[label] = distance
            
            # Pilih label dengan jarak terdekat
            if distances:  # Pastikan distances tidak kosong
                closest_label = min(distances, key=distances.get)
                cluster_mapping[cluster_idx] = closest_label
    
    print(f"🔀 Mapping cluster: {cluster_mapping}")
    
    df_features['cluster_label'] = df_features['cluster'].map(cluster_mapping)
    
    # Simpan informasi centroid yang digunakan
    centroid_info = []
    for cluster_idx, label in cluster_mapping.items():
        centroid_info.append({
            'cluster': cluster_idx,
            'cluster_label': label,
            'centroid_values': centroids[cluster_idx] if cluster_idx < len(centroids) else None
        })
    
    return df_features, kmeans, feature_columns, centroid_info

def analyze_clustering_results_10classes(df_features, df_conversion, centroid_info):
    """
    Menganalisis dan menampilkan hasil clustering untuk 10 kelas
    """
    print("\n" + "="*80)
    print("HASIL K-MEANS CLUSTERING - 10 KELAS CIFAR-10")
    print("="*80)
    
    # Tampilkan informasi centroid
    print("\n🎯 INFORMASI CENTROID:")
    for i, centroid in enumerate(centroid_info):
        print(f"   Cluster {centroid['cluster']}: {centroid['cluster_label']}")
    
    # Hitung akurasi clustering
    correct_predictions = 0
    total_predictions = len(df_features)
    
    clustering_results = []
    
    for _, row in df_features.iterrows():
        actual_label = row['label']
        predicted_label = row['cluster_label']
        is_correct = (actual_label == predicted_label)
        
        if is_correct:
            correct_predictions += 1
        
        clustering_results.append({
            'filename': row['filename'],
            'gambar_ke': row['gambar_ke'],
            'label_aktual': actual_label,
            'cluster_prediksi': row['cluster'],
            'label_prediksi': predicted_label,
            'benar': is_correct,
            'jarak_ke_centroid': row['distance_to_centroid']
        })
    
    accuracy = correct_predictions / total_predictions * 100 if total_predictions > 0 else 0
    
    print(f"\n📈 AKURASI CLUSTERING: {accuracy:.2f}%")
    print(f"   Benar: {correct_predictions}/{total_predictions}")
    
    # Tampilkan confusion matrix sederhana
    print("\n📊 CONFUSION MATRIX:")
    df_results = pd.DataFrame(clustering_results)
    if not df_results.empty:
        confusion_matrix = pd.crosstab(df_results['label_aktual'], df_results['label_prediksi'])
        print(confusion_matrix)
    
    return df_results, accuracy

def plot_clustering_results_10classes(df_features, feature_columns, centroid_info):
    """
    Membuat visualisasi hasil clustering untuk 10 kelas
    """
    print("\n" + "="*80)
    print("VISUALISASI HASIL CLUSTERING - 10 KELAS CIFAR-10")
    print("="*80)
    
    # Standardisasi fitur
    X = df_features[feature_columns].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Reduksi dimensi dengan PCA
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    
    # Hitung batas yang sama untuk kedua plot
    x_min = min(X_pca[:, 0]) - 0.5
    x_max = max(X_pca[:, 0]) + 0.5
    y_min = min(X_pca[:, 1]) - 0.5
    y_max = max(X_pca[:, 1]) + 0.5
    
    # Buat plot dengan ukuran yang lebih besar
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
    
    # Warna untuk plot
    colors = {
        'airplane': 'blue', 'automobile': 'red', 'bird': 'green', 'cat': 'orange', 
        'deer': 'purple', 'dog': 'brown', 'frog': 'pink', 'horse': 'gray', 
        'ship': 'olive', 'truck': 'cyan'
    }
    
    # Plot 1: Label aktual
    actual_labels = df_features['label'].unique()
    for label in actual_labels:
        mask = df_features['label'] == label
        color = colors.get(label, 'black')  # Default black jika label tidak dikenali
        if mask.any():
            ax1.scatter(X_pca[mask, 0], X_pca[mask, 1], 
                       c=color, label=f'Aktual: {label}', s=100, alpha=0.7, edgecolors='black', linewidth=0.5)
    
    ax1.set_title('LABEL AKTUAL (Ground Truth) - CIFAR-10', fontsize=14, fontweight='bold')
    ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
    ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(x_min, x_max)
    ax1.set_ylim(y_min, y_max)
    
    # Plot 2: Hasil clustering
    cluster_labels = df_features['cluster_label'].unique()
    for cluster_label in cluster_labels:
        mask = df_features['cluster_label'] == cluster_label
        color = colors.get(cluster_label, 'black')  # Default black jika label tidak dikenali
        if mask.any():
            ax2.scatter(X_pca[mask, 0], X_pca[mask, 1], 
                       c=color, label=f'Cluster: {cluster_label}', s=100, alpha=0.7, edgecolors='black', linewidth=0.5)
    
    # Tandai centroid dengan spesial marker
    print(f"🎯 Menampilkan {len(centroid_info)} centroid di plot...")
    centroid_markers = ['*', 'D', 's', '^', 'v', '<', '>', 'p', 'h', 'X']
    centroid_sizes = [400, 300, 350, 450, 500, 320, 380, 420, 470, 530]
    
    # Plot centroid (gunakan PCA transform dari centroid asli)
    for idx, centroid in enumerate(centroid_info):
        if centroid['centroid_values'] is not None:
            # Transform centroid ke ruang PCA
            centroid_scaled = scaler.transform([centroid['centroid_values']])
            centroid_pca = pca.transform(centroid_scaled)[0]
            
            marker = centroid_markers[idx % len(centroid_markers)]
            size = centroid_sizes[idx % len(centroid_sizes)]
            
            ax2.scatter(centroid_pca[0], centroid_pca[1], 
                       c='black', marker=marker, s=size,
                       label=f"Centroid: {centroid['cluster_label']}",
                       edgecolors='white', linewidth=2, zorder=5)
            
            print(f"   ✅ Centroid {idx}: {centroid['cluster_label']} di posisi {centroid_pca}")
    
    ax2.set_title('HASIL CLUSTERING K-MEANS - CIFAR-10', fontsize=14, fontweight='bold')
    ax2.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
    ax2.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
    ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(x_min, x_max)
    ax2.set_ylim(y_min, y_max)
    
    plt.tight_layout()
    plt.show()
    
    # Print variance explained oleh PCA
    print(f"\n📊 Variance yang dijelaskan oleh PCA:")
    print(f"   PC1: {pca.explained_variance_ratio_[0]:.4f} ({pca.explained_variance_ratio_[0]:.2%})")
    print(f"   PC2: {pca.explained_variance_ratio_[1]:.4f} ({pca.explained_variance_ratio_[1]:.2%})")
    print(f"   Total: {sum(pca.explained_variance_ratio_):.4f} ({sum(pca.explained_variance_ratio_):.2%})")

def save_clustering_results(df_features, df_results, accuracy, output_path):
    """
    Menyimpan hasil clustering ke file Excel
    """
    with pd.ExcelWriter(output_path) as writer:
        # Sheet 1: Fitur dan hasil clustering
        df_features.to_excel(writer, sheet_name='Fitur_dan_Clustering', index=False)
        
        # Sheet 2: Ringkasan hasil
        df_results.to_excel(writer, sheet_name='Hasil_Clustering', index=False)
        
        # Sheet 3: Statistik clustering
        accuracy_data = [{
            'total_gambar': len(df_features),
            'benar': len(df_results[df_results['benar'] == True]),
            'salah': len(df_results[df_results['benar'] == False]),
            'akurasi': accuracy
        }]
        df_accuracy = pd.DataFrame(accuracy_data)
        df_accuracy.to_excel(writer, sheet_name='Statistik', index=False)
    
    print(f"💾 Hasil clustering disimpan ke: {output_path}")

# =============================================
# FUNGSI UTAMA UNTUK CIFAR-10
# =============================================

def main_cifar10_training():
    """
    Fungsi utama untuk memproses data training CIFAR-10 (10 kelas)
    """
    print("PROSES EKSTRAKSI RGB, ECDF, DAN ANALISIS INTENSITAS - 10 KELAS CIFAR-10 (TRAINING)")
    print("=" * 80)
    
    # 1. Proses gambar training dari 10 kelas CIFAR-10
    print("\n1. PROSES GAMBAR TRAINING DARI 10 KELAS CIFAR-10")
    df_train = process_cifar10_dataset(cifar10_train_folder, "training")
    
    if df_train.empty:
        print("❌ Tidak ada data training yang diproses. Periksa path folder dan file gambar.")
        return None, None
    
    print(f"\n📊 Total gambar training diproses: {len(df_train)}")
    for class_name in cifar10_classes:
        count = len(df_train[df_train['label'] == class_name])
        print(f"  - {class_name}: {count} gambar")
    
    # 2. Simpan data RGB training ke Excel
    print(f"\n2. MENYIMPAN DATA RGB TRAINING KE EXCEL: {output_excel}")
    save_rgb_to_excel(df_train, output_excel)
    
    # 3. Hitung ECDF untuk semua gambar training
    print("\n3. MENGHITUNG ECDF UNTUK SEMUA GAMBAR TRAINING")
    train_ecdf_results = calculate_all_ecdf(df_train)
    
    # 4. Simpan data ECDF training ke CSV
    print(f"\n4. MENYIMPAN DATA ECDF TRAINING KE CSV: {ecdf_output_csv}")
    save_ecdf_to_csv(train_ecdf_results, ecdf_output_csv)
    
    # 5. Ekstrak nilai ECDF pada intensitas tertentu dari training
    print("\n5. EKSTRAKSI NILAI ECDF PADA INTENSITAS 100, 150, 200 DARI TRAINING")
    df_ecdf_intensities = extract_ecdf_at_intensities(train_ecdf_results)
    
    # 6. Hitung CDF untuk nilai ECDF pada intensitas tertentu dari training
    print("\n6. MENGHITUNG CDF UNTUK NILAI ECDF PADA INTENSITAS TERTENTU DARI TRAINING")
    df_cdf = calculate_cdf_for_ecdf_intensities(df_ecdf_intensities)
    
    # 7. Simpan data ECDF intensitas dan CDF training
    print(f"\n7. MENYIMPAN DATA ECDF INTENSITAS DAN CDF TRAINING: {ecdf_intensity_output_excel}")
    save_ecdf_intensity_data(df_ecdf_intensities, df_cdf, ecdf_intensity_output_excel)
    
    # 8. Membuat model matematis V dari data CDF training
    print("\n8. MEMBUAT MODEL MATEMATIS V DARI DATA CDF INTENSITAS TRAINING")
    model_V = create_mathematical_model_V(df_cdf)
    
    # 9. Ekstrak ECDF dari gambar training dan konversi ke CDF menggunakan model V
    print("\n9. EKSTRAKSI ECDF DARI GAMBAR TRAINING DAN KONVERSI KE CDF MENGGUNAKAN MODEL V")
    df_conversion = extract_and_convert_ecdf_data(train_ecdf_results, model_V)
    
    # 10. Simpan data konversi CDF training
    print(f"\n10. MENYIMPAN DATA KONVERSI CDF TRAINING: {conversion_output_excel}")
    save_conversion_data(train_ecdf_results, df_conversion, conversion_output_excel)
    
    # 11. Persiapan fitur untuk clustering dari CDF hasil konversi training
    print("\n11. PERSIAPAN FITUR UNTUK CLUSTERING DARI CDF HASIL KONVERSI TRAINING")
    df_features = prepare_features_for_clustering(df_conversion)
    
    if df_features.empty:
        print("❌ Tidak ada fitur yang dapat dipersiapkan untuk clustering!")
        return None, None
    
    print(f"   Jumlah gambar training untuk clustering: {len(df_features)}")
    print(f"   Jumlah fitur per gambar: {len([col for col in df_features.columns if col not in ['filename', 'gambar_ke', 'label']])}")
    
    # 12. Hitung centroid dari data training
    print("\n12. MENGHITUNG CENTROID DARI DATA TRAINING")
    centroids_dict, feature_columns = calculate_class_centroids(df_features)
    
    # 13. Simpan centroid ke file
    print(f"\n13. MENYIMPAN CENTROID: {centroid_output_file}")
    save_centroids(centroids_dict, feature_columns, centroid_output_file)
    
    print("\n" + "="*80)
    print("🎉 PROSES TRAINING SELESAI UNTUK 10 KELAS CIFAR-10!")
    print("="*80)
    
    return model_V, centroids_dict

def main_cifar10_testing(model_V, centroids_dict):
    """
    Fungsi utama untuk memproses data testing CIFAR-10 (10 kelas)
    """
    print("\n\nPROSES TESTING DAN CLUSTERING - 10 KELAS CIFAR-10 (TESTING)")
    print("=" * 80)
    
    # 1. Proses gambar testing dari 10 kelas CIFAR-10
    print("\n1. PROSES GAMBAR TESTING DARI 10 KELAS CIFAR-10")
    df_test = process_cifar10_dataset(cifar10_test_folder, "testing")
    
    if df_test.empty:
        print("❌ Tidak ada data testing yang diproses. Periksa path folder dan file gambar.")
        return
    
    print(f"\n📊 Total gambar testing diproses: {len(df_test)}")
    for class_name in cifar10_classes:
        count = len(df_test[df_test['label'] == class_name])
        print(f"  - {class_name}: {count} gambar")
    
    # 2. Hitung ECDF untuk semua gambar testing
    print("\n2. MENGHITUNG ECDF UNTUK SEMUA GAMBAR TESTING")
    test_ecdf_results = calculate_all_ecdf(df_test)
    
    # 3. Ekstrak ECDF dari gambar testing dan konversi ke CDF menggunakan model V dari training
    print("\n3. EKSTRAKSI ECDF DARI GAMBAR TESTING DAN KONVERSI KE CDF MENGGUNAKAN MODEL V")
    df_conversion_test = extract_and_convert_ecdf_data(test_ecdf_results, model_V)
    
    # 4. Persiapan fitur untuk clustering dari CDF hasil konversi testing
    print("\n4. PERSIAPAN FITUR UNTUK CLUSTERING DARI CDF HASIL KONVERSI TESTING")
    df_features_test = prepare_features_for_clustering(df_conversion_test)
    
    if df_features_test.empty:
        print("❌ Tidak ada fitur yang dapat dipersiapkan untuk clustering testing!")
        return
    
    print(f"   Jumlah gambar testing untuk clustering: {len(df_features_test)}")
    
    # 5. Melakukan K-means clustering dengan centroid dari training
    print("\n5. MELAKUKAN K-MEANS CLUSTERING DENGAN CENTROID DARI TRAINING")
    feature_columns = [col for col in df_features_test.columns if col not in ['filename', 'gambar_ke', 'label']]
    
    df_features_test, kmeans, feature_columns, centroid_info = perform_kmeans_clustering_10centroids(
        df_features_test, centroids_dict, feature_columns
    )
    
    if df_features_test is None:
        print("❌ Clustering testing gagal!")
        return
    
    # 6. Analisis hasil clustering testing
    print("\n6. ANALISIS HASIL CLUSTERING TESTING")
    df_results, accuracy = analyze_clustering_results_10classes(df_features_test, df_conversion_test, centroid_info)
    
    # 7. Visualisasi hasil clustering testing
    print("\n7. VISUALISASI HASIL CLUSTERING TESTING")
    plot_clustering_results_10classes(df_features_test, feature_columns, centroid_info)
    
    # 8. Simpan hasil clustering testing
    print(f"\n8. MENYIMPAN HASIL CLUSTERING TESTING: {clustering_output_excel}")
    save_clustering_results(df_features_test, df_results, accuracy, clustering_output_excel)
    
    print("\n" + "="*80)
    print("🎉 PROSES TESTING SELESAI UNTUK 10 KELAS CIFAR-10!")
    print("="*80)
    
    # Ringkasan akhir
    print("\n📊 RINGKASAN HASIL TESTING:")
    print(f"   - Total gambar testing diproses: {len(df_test)}")
    print(f"   - Akurasi clustering: {accuracy:.2f}%")
    print(f"   - File output:")
    print(f"     * Data RGB Training: {output_excel}")
    print(f"     * Data ECDF Training (CSV): {ecdf_output_csv}")
    print(f"     * Data ECDF Intensitas Training: {ecdf_intensity_output_excel}")
    print(f"     * Data Konversi CDF Training: {conversion_output_excel}")
    print(f"     * Centroid: {centroid_output_file}")
    print(f"     * Hasil Clustering Testing: {clustering_output_excel}")

def main_cifar10_full():
    """
    Fungsi utama untuk menjalankan proses lengkap CIFAR-10
    """
    # Jalankan proses training
    model_V, centroids_dict = main_cifar10_training()
    
    if model_V is None or centroids_dict is None:
        print("❌ Proses training gagal!")
        return
    
    # Jalankan proses testing
    main_cifar10_testing(model_V, centroids_dict)

if __name__ == "__main__":
    # Jalankan proses lengkap CIFAR-10
    main_cifar10_full()

PROSES EKSTRAKSI RGB, ECDF, DAN ANALISIS INTENSITAS - 10 KELAS CIFAR-10 (TRAINING)

1. PROSES GAMBAR TRAINING DARI 10 KELAS CIFAR-10
📁 Memproses 5000 gambar dari C:/Users/Dhea/Downloads/archive\cifar10_train\airplane (label: airplane)
✅ Berhasil memproses 5000 gambar dengan label: airplane
📁 Memproses 5000 gambar dari C:/Users/Dhea/Downloads/archive\cifar10_train\automobile (label: automobile)
✅ Berhasil memproses 5000 gambar dengan label: automobile
📁 Memproses 5000 gambar dari C:/Users/Dhea/Downloads/archive\cifar10_train\bird (label: bird)
✅ Berhasil memproses 5000 gambar dengan label: bird
📁 Memproses 5000 gambar dari C:/Users/Dhea/Downloads/archive\cifar10_train\cat (label: cat)
✅ Berhasil memproses 5000 gambar dengan label: cat
📁 Memproses 5000 gambar dari C:/Users/Dhea/Downloads/archive\cifar10_train\deer (label: deer)
✅ Berhasil memproses 5000 gambar dengan label: deer
📁 Memproses 5000 gambar dari C:/Users/Dhea/Downloads/archive\cifar10_train\dog (label: dog)
✅ Berhasil mempros

MemoryError: Unable to allocate 715. MiB for an array with shape (93750000,) and data type int64